In [331]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import train_test_split
import csv

In [332]:
def print_dir(obj):
    print((f"---| {obj.__class__} |---"))
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [333]:
path = "D:\\projects\\Supervised-Learning-Experiments\\olympiads\\competition_samples\\raw\\romania-onia-examples\\Solutii\\Candidat_incepator"
train_df = pd.read_csv(f"{path}\\dataset_train.csv")
eval_df = pd.read_csv(f"{path}\\dataset_eval.csv")

In [334]:
train_df

,AppID,Name,Release date,Estimated owners,Price,Metacritic score,Recommendations,Positive,Negative,Publishers,Genres
0,350110,TransOcean 2: Rivals,"May 10, 2016",100000 - 200000,29.99,69,428,305,281,astragon Entertainment,"Simulation,Strategy"
1,312780,Way of the Samurai 4,"July 23, 2015",200000 - 500000,24.99,72,1475,1288,318,"Spike Chunsoft Co., Ltd.","Action,Adventure"
2,230150,Incredipede,"March 18, 2013",50000 - 100000,9.99,74,140,243,96,Northway Games,"Action,Adventure,Casual,Indie,Simulation"
3,1252300,RetroMania Wrestling,"February 25, 2021",0 - 20000,29.99,66,174,211,30,Retrosoft Studios,"Action,Sports"
4,760650,Hammerting,"November 16, 2021",100000 - 200000,9.99,64,1358,1328,705,Team17 Digital,"Adventure,Indie,RPG,Simulation,Strategy"
...,...,...,...,...,...,...,...,...,...,...,...
2995,266410,iRacing,"January 12, 2015",100000 - 200000,9.99,79,1281,2096,334,"iRacing.com Motorsport Simulations,iRacing","Massively Multiplayer,Racing,Simulation,Sports"
2996,939100,Darksburg,"September 23, 2020",100000 - 200000,14.99,61,1489,1511,750,Shiro Unlimited,"Action,Indie"
2997,1931770,Chants of Sennaar,"September 05, 2023",100000 - 200000,19.99,85,6125,6484,82,Focus Entertainment,"Adventure,Indie"
2998,41500,Torchlight,"October 27, 2009",1000000 - 2000000,14.99,83,4323,5424,495,Runic Games,RPG


In [335]:
eval_df

,AppID,Name,Release date,Estimated owners,Metacritic score,Recommendations,Positive,Negative,Publishers,Genres
0,636230,Safe House,"May 22, 2018",0 - 20000,46,0,13,19,Labs Games,"Adventure,Indie,Strategy"
1,256460,Cosmic Star Heroine,"April 11, 2017",100000 - 200000,77,574,677,82,Zeboyd Games,"Indie,RPG"
2,327890,I Am Bread,"April 09, 2015",500000 - 1000000,60,4525,4359,1247,Bossa Studios,"Action,Adventure,Indie,Simulation"
3,287020,Harvester,"April 04, 2014",50000 - 100000,53,769,905,88,Nightdive Studios,Adventure
4,25980,Majesty 2,"September 17, 2009",200000 - 500000,72,783,611,270,Paradox Interactive,"Simulation,Strategy"
...,...,...,...,...,...,...,...,...,...,...
595,269650,Dex,"May 07, 2015",200000 - 500000,62,2469,2655,389,"Dreadlocks Ltd.,Techland,WhisperGames","Action,Adventure,Indie,RPG"
596,261900,The Real Texas,"July 12, 2016",0 - 20000,74,0,28,2,Kitty Lambda Games Inc.,"Action,Adventure,Indie,RPG"
597,209630,Magrunner: Dark Pulse,"June 20, 2013",200000 - 500000,70,314,702,254,Frogwares,"Action,Adventure,Indie"
598,578650,The Outer Worlds,"October 23, 2020",500000 - 1000000,82,17319,16925,2891,Private Division,RPG


In [336]:
# print_dir(np.ndarray)

In [337]:
train_df_numpy = train_df.to_numpy()
samples = len(train_df)
print(samples)
average_price = int(train_df["Price"].mean())
print(average_price)
owners = np.array([(int(j[1]) + int(j[0]))//2 for j in [i.split("-") for i in train_df["Estimated owners"]]])
eval_owners = np.array([(int(j[1]) + int(j[0]))//2 for j in [i.split("-") for i in eval_df["Estimated owners"]]])
average_owners = sum(owners)//len(train_df)
print(average_owners)

unique_gernes = []
for gernes in train_df["Genres"]:
    for gerne in gernes.split(","):
        unique_gernes.append(gerne)
unique_gernes_count = len(np.unique(np.array(unique_gernes)))
print(unique_gernes_count)

3000
16
628183
16


In [338]:
X_eval = np.concatenate((
    eval_owners.reshape(-1, 1), 
    eval_df["Metacritic score"].to_numpy().reshape(-1,1),
    eval_df["Recommendations"].to_numpy().reshape(-1,1),
    eval_df["Positive"].to_numpy().reshape(-1,1),
    eval_df["Negative"].to_numpy().reshape(-1,1),
    ), axis=1)
X = np.concatenate((
    owners.reshape(-1, 1), 
    train_df["Metacritic score"].to_numpy().reshape(-1,1),
    train_df["Recommendations"].to_numpy().reshape(-1,1),
    train_df["Positive"].to_numpy().reshape(-1,1),
    train_df["Negative"].to_numpy().reshape(-1,1),
    ), axis=1)
y = train_df["Price"].to_numpy()
print(X)
print(y.dtype)

[[ 150000      69     428     305     281]
 [ 350000      72    1475    1288     318]
 [  75000      74     140     243      96]
 ...
 [ 150000      85    6125    6484      82]
 [1500000      83    4323    5424     495]
 [ 350000      75    3268    3723     305]]
float64


In [339]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=.2)

In [340]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train);
y_pred_test = model.predict(X_test)
loss = np.abs(y_test-y_pred_test).mean()
print(loss)

7.598781166666664


In [341]:
model.fit(X, y);
y_eval = model.predict(X_eval).tolist()
print(y_eval)

[9.110000000000007, 9.333600000000006, 13.998000000000006, 11.734600000000007, 11.018500000000008, 10.955000000000007, 13.540000000000006, 13.680000000000007, 14.290100000000006, 10.290000000000006, 20.10000000000001, 16.580000000000005, 9.702000000000007, 18.577500000000008, 18.960000000000008, 14.571000000000009, 7.605, 14.060000000000008, 12.284500000000007, 13.785000000000007, 20.470000000000006, 17.577500000000008, 14.470000000000006, 15.820000000000007, 15.921000000000008, 17.905500000000007, 12.610000000000007, 18.880000000000006, 14.087500000000007, 11.197000000000008, 18.868100000000005, 12.071000000000005, 15.420000000000007, 13.372000000000007, 9.945000000000007, 20.862500000000004, 19.795000000000005, 9.800000000000006, 11.671400000000006, 8.998600000000005, 23.079999999999988, 24.309999999999974, 20.595500000000005, 8.640000000000008, 18.91380000000001, 13.910000000000007, 12.900000000000007, 7.870000000000005, 9.478100000000007, 41.82499999999989, 33.47999999999992, 20.14

In [342]:
with open("output_1.csv", "w", newline="") as csvfile:
    fieldnames = "Samples","Average Price","Average Owners","Unique Genres"
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows([{"Samples":samples, "Average Price":average_price, "Average Owners":average_owners, "Unique Genres":unique_gernes_count}])

In [343]:
with open("output_2.csv", "w", newline="") as csvfile:
    fieldnames = ["Price"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i in y_eval:
        writer.writerows([{"Price":i}])